# Lezione 3 — Supremazia quantistica e simulazione classica

## Obiettivi di apprendimento

- Spiegare in cosa consiste la supremazia quantistica e come si verifica con il benchmarking a entropia incrociata (XEB)
- Usare la trotterizzazione per approssimare l'evoluzione temporale di un sistema quantistico
- Mettere in relazione la profondità di un circuito con la precisione della simulazione

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum

## Perché è importante

Nel 2019 Google annunciò che il suo processore Sycamore aveva risolto in 200 secondi un calcolo che, secondo le sue stime, avrebbe richiesto migliaia di anni al supercomputer classico più potente del mondo. Questo traguardo si chiama **supremazia quantistica**: il momento in cui un computer quantistico risolve un compito che nessun computer classico può completare in un tempo ragionevole. Se capisci perché quel calcolo è così difficile da simulare classicamente, e come si verifica che il risultato quantistico sia corretto, hai l'intuizione chiave di tutto il corso: la potenza del calcolo quantistico cresce in modo esponenziale con il numero di qubit, e il costo di simularlo su un computer classico cresce altrettanto rapidamente.

## Il costo esponenziale di simulare un circuito quantistico

Un sistema di n qubit in sovrapposizione richiede 2^n numeri per essere descritto completamente. Con 53 qubit, come quelli di Sycamore, ottieni più di 9.000 miliardi di numeri: non entrano più nella memoria di nessun computer classico esistente. Più qubit aggiungi al tuo circuito, più rapidamente esplode il costo della sua simulazione, mentre il computer quantistico esegue quel circuito senza questo costo aggiuntivo: letteralmente, è quel sistema quantistico.

## Campionamento di circuiti casuali: il compito risolto da Sycamore

Il compito che dimostra la supremazia quantistica non è risolvere un problema utile, ma qualcosa di più sottile: generare campioni della distribuzione di probabilità in uscita di un circuito quantistico casuale e profondo. Un computer quantistico genera quei campioni in modo naturale, semplicemente eseguendo il circuito più volte. Un computer classico, invece, deve calcolare quella distribuzione moltiplicando matrici enormi, e questo calcolo diventa impraticabile a partire da un certo numero di qubit e di profondità del circuito.

## Come verifichi che il risultato quantistico sia corretto: XEB

Se nessuno può calcolare classicamente l'intera distribuzione di probabilità, come fai a sapere che i campioni del computer quantistico sono corretti? La risposta è il benchmarking a entropia incrociata (XEB): invece di calcolare la distribuzione completa, calcoli la probabilità teorica di ciascun campione concreto che hai osservato (molto meno costoso che calcolare tutte le probabilità possibili) e la confronti con ciò che ti aspetteresti da un risultato puramente casuale. Più i tuoi campioni osservati assomigliano alla distribuzione teorica del circuito, e non al rumore casuale, più hai fiducia che l'hardware funzioni davvero come un computer quantistico.

La forma che assume quella distribuzione teorica in uscita —molte probabilità basse e poche alte, come macchie di luce laser sparpagliate— è nota come distribuzione di Porter-Thomas, ed è l'impronta digitale che cerchi confrontando i risultati reali con la teoria. Una tecnica correlata, il benchmarking della purezza delle macchie (speckle purity benchmarking), ti aiuta a separare quanto di quell'impronta è segnale quantistico reale e quanto è rumore dell'hardware.

## Quantificare la fiducia: significatività statistica

Affermare di aver dimostrato la supremazia quantistica è un'affermazione forte, quindi devi sostenerla con una statistica rigorosa. Puoi usare strumenti come il test di Kolmogorov-Smirnov (KS), che verifica se due distribuzioni di dati sono statisticamente indistinguibili, e tecniche di bootstrapping (ricampionamento ripetuto degli stessi dati) per stimare quanto variano le tue conclusioni se ripetessi l'esperimento. Il risultato finale si esprime di solito in livelli sigma: più deviazioni standard di margine hai rispetto all'ipotesi nulla —cioè che il risultato sia rumore casuale—, più solida è la tua conclusione.

## L'era NISQ: potenti, ma imperfetti

Sycamore, Bristlecone (il suo predecessore) e tutti i processori quantistici attuali appartengono all'era NISQ (quantistico a scala intermedia e rumoroso, della seconda parte della lezione 1): hanno decine o qualche centinaio di qubit, e il rumore limita la profondità dei circuiti che puoi eseguire in modo affidabile. La supremazia quantistica dimostra che, anche in questa era imperfetta, esistono già compiti che superano il calcolo classico. La sfida delle prossime lezioni è diversa: non dimostrare la supremazia con circuiti casuali, ma usare quegli stessi computer rumorosi per risolvere problemi utili di machine learning, chimica e ottimizzazione.

## Trotterizzazione: simulare l'evoluzione di un sistema quantistico

Un modo pratico per percepire il costo esponenziale è simulare l'evoluzione nel tempo di un sistema quantistico. Calcolare questa evoluzione in modo esatto richiede di moltiplicare matrici che crescono come 2^n, quindi in pratica la approssimi dividendo l'evoluzione in piccoli passi: questa tecnica si chiama trotterizzazione. Più passi di Trotter usi, più ti avvicini all'evoluzione esatta del sistema, ma devi anche concatenare più porte quantistiche, e ricorda la lezione 1: più porte significano più tempo esposto alla decoerenza. Questo equilibrio tra precisione e profondità del circuito è esattamente lo stesso dilemma che affronta qualsiasi computer quantistico NISQ reale.

Di seguito applichi questa idea a un esempio concreto: una catena di spin a quattro qubit con campo trasverso, un sistema tipico della fisica dei materiali.

### Definisci l'Hamiltoniano della catena di Ising con campo trasverso

Ogni qubit interagisce con il suo vicino tramite un termine Z-Z, e inoltre risente di un proprio campo trasverso X.

In [ ]:
import cirq
import tensorflow_quantum as tfq
import numpy as np
import matplotlib.pyplot as plt

def cadena_ising_campo_transverso(n, campos):
    """Hamiltoniano de una cadena de n qubits con interacción Z-Z y campo X."""
    qubits = [cirq.GridQubit(0, i) for i in range(n)]
    hamiltoniano = cirq.PauliSum()
    for i in range(n - 1):
        hamiltoniano -= cirq.Z(qubits[i]) * cirq.Z(qubits[i + 1])
    for i in range(n):
        hamiltoniano -= campos[i] * cirq.X(qubits[i])
    return hamiltoniano

### Definisci la trotterizzazione: un passo e la ripetizione completa

`paso_trotter` applica ciascun termine dell'Hamiltoniano separatamente, come approssimazione di un singolo passo. `trotter_lie` concatena più di questi passi: più ripetizioni usi, più fine è ciascun passo e più preciso è il risultato finale.

In [ ]:
def paso_trotter(hamiltoniano, t):
    """Aproxima exp(-i * H * t) aplicando cada término del Hamiltoniano por separado."""
    return cirq.Circuit([tfq.util.exponential([t * termino]) for termino in hamiltoniano])

def trotter_lie(hamiltoniano, t, repeticiones):
    """Cuantas más repeticiones, más fino el paso y más preciso el resultado."""
    circuito = cirq.Circuit()
    for _ in range(repeticiones):
        circuito += paso_trotter(hamiltoniano, t / repeticiones)
    return circuito

### Costruisci i circuiti e calcola la magnetizzazione con TensorFlow Quantum

Confronti due versioni dello stesso esperimento: una con un solo passo di Trotter (approssimazione grossolana) e un'altra con dieci passi (approssimazione fine). In entrambi i casi misuri la magnetizzazione totale della catena, cioè la somma degli operatori Z di ciascun qubit.

In [ ]:
n = 4
campos = [1.0] * n
hamiltoniano = cadena_ising_campo_transverso(n, campos)
qubits = [cirq.GridQubit(0, i) for i in range(n)]
magnetizacion = sum(cirq.Z(q) for q in qubits)

tiempos = np.linspace(0.01, 2, 20)

# Con solo 1 repetición, el paso de trotter es "grueso" y menos preciso
circuitos_1_paso = [trotter_lie(hamiltoniano, float(t), 1) for t in tiempos]
# Con 10 repeticiones, el paso es más fino y se acerca más a la evolución real
circuitos_10_pasos = [trotter_lie(hamiltoniano, float(t), 10) for t in tiempos]

capa_expectativa = tfq.layers.Expectation()
exp_1_paso = capa_expectativa(circuitos_1_paso, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()
exp_10_pasos = capa_expectativa(circuitos_10_pasos, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()

### Visualizza il risultato

Disegna le due curve di magnetizzazione e osserva quanto si allontanano l'una dall'altra man mano che passa il tempo.

In [ ]:
plt.plot(tiempos, exp_1_paso, label="1 paso de Trotter")
plt.plot(tiempos, exp_10_pasos, label="10 pasos de Trotter")
plt.xlabel("Tiempo")
plt.ylabel("Magnetización")
plt.legend()
plt.show()

La curva con un passo si allontana sempre di più dalla curva di riferimento al progredire del tempo, perché l'errore dell'approssimazione si accumula a ogni passo. La curva con dieci passi resta molto più vicina all'evoluzione reale, al prezzo di un circuito con molte più porte concatenate.

## Riepilogo dei punti chiave

- I ricercatori hanno dimostrato la supremazia quantistica con un compito di campionamento di circuiti casuali, impraticabile da simulare su un computer classico a causa della crescita esponenziale (2^n) degli stati possibili
- Il benchmarking a entropia incrociata (XEB) ti permette di verificare che i campioni quantistici osservati corrispondano alla distribuzione teorica del circuito (la distribuzione di Porter-Thomas) e non al rumore
- Quantifichi la fiducia statistica nel risultato con test come Kolmogorov-Smirnov, bootstrapping e livelli sigma
- Viviamo nell'era NISQ: computer quantistici reali ma rumorosi, con pochi qubit e circuiti poco profondi
- La trotterizzazione approssima l'evoluzione di un sistema quantistico dividendola in piccoli passi: più passi significano più precisione, ma anche più profondità del circuito e più esposizione al rumore

## Glossario degli acronimi

- KS — test di Kolmogorov-Smirnov
- NISQ — quantistico a scala intermedia e rumoroso (Noisy Intermediate-Scale Quantum)
- XEB — benchmarking a entropia incrociata (Cross-Entropy Benchmarking)